In [64]:
import pyvisa
from typing import TYPE_CHECKING, Callable
from qcodes.instrument import find_or_create_instrument
from qblox_instruments import Cluster, ClusterType
from qblox_instruments.qcodes_drivers.module import Module

!qblox-pnp list

Devices:
 - 129.97.9.55: cluster_mm 0.10.0 with name "cluster-mm" and serial number 00015_2251_003


In [75]:
cluster_ip = "129.97.9.55"
cluster_name = "cluster0"
cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=(
        {
            2: ClusterType.CLUSTER_QCM,
            4: ClusterType.CLUSTER_QRM,
            6: ClusterType.CLUSTER_QCM_RF,
        }
        if cluster_ip is None
        else None
    ),
)
cluster.led_brightness('medium') # Sets LED brightness on the modules. Options are 'low', 'medium' and 'high'
def get_connected_modules(cluster: Cluster, filter_fn: Callable | None = None) -> dict[int, Module]:
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True
    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }
modules = get_connected_modules(cluster)
module = list(modules.values())[0]
cluster.reset()
print(cluster.get_system_status())
# Assigning modules
qrm_module = modules[4]
qcm_module = modules[2]
rf_module = modules[2]
# Connecting to the osciloscope
rm = pyvisa.ResourceManager()
scope = rm.open_resource('GPIB0::8::INSTR')

Status: OKAY, Flags: NONE, Slot flags: NONE


In [76]:
# Stopping and resetting
cluster.stop_sequencer()
print("RF sequencer 0:  " + str(rf_module.get_sequencer_status(0)))
cluster.reset()
print(cluster.get_system_status())

RF sequencer 0:  Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, Flags: NONE, Slot flags: NONE


In [78]:

import scipy.signal
# Waveform parameters
waveform_length = 100  # nanoseconds

# Waveform dictionary (data will hold the samples and index will be used to select the waveforms in the instrument).
waveforms = {
    "gaussian": {
        "data": scipy.signal.windows.gaussian(waveform_length, std=0.12 * waveform_length).tolist(),
        "index": 0,
    },
    "block": {"data": [1.0 for i in range(0, waveform_length)], "index": 1},
}

In [79]:
seq_prog = """
        move      100,R0   #Loop iterator.
        move      20,R1    #Initial wait period in ns.
        wait_sync 4        #Wait for sequencers to synchronize and then wait another 4 ns.

        
        play      1,1,22    #Play a gaussian and a block on output path 0 and 1 respectively and wait 4 ns.

        stop               #Stop the sequence after the last iteration.
"""


# Upload sequence
rf_module.sequencer0.sequence(
    {
        "waveforms": waveforms,
        "weights": {},
        "acquisitions": {},
        "program": seq_prog,
    }
)

# Upload sequence
rf_module.sequencer1.sequence(
    {
        "waveforms": waveforms,
        "weights": {},
        "acquisitions": {},
        "program": seq_prog,
    }
)


In [86]:
rf_module.sequencer0.sync_en(True)
rf_module.sequencer1.sync_en(True)

rf_module.disconnect_outputs()

rf_module.sequencer0.connect_sequencer("out0")
rf_module.sequencer1.connect_sequencer("out1")

""" rf_module.sequencer0.marker_ovr_en(True)
rf_module.sequencer0.marker_ovr_value(3)  # Enables output on ch0

rf_module.sequencer1.marker_ovr_en(True)
rf_module.sequencer1.marker_ovr_value(3)  # Enables output on ch1

rf_module.sequencer0.mod_en_awg(True)
rf_module.sequencer1.mod_en_awg(True)  # Enables output on ch1 """

""" # module setup
rf_module.out0_offset_path0(0)
rf_module.out0_offset_path1(0)
rf_module.out1_offset_path0(0)
rf_module.out1_offset_path1(0)

#rf_module.out0_lo_en(True)
#rf_module.out1_lo_en(True)

base = 2.1e9
diff = 0.05e9

rf_module.out0_lo_freq(base+diff)
rf_module.out1_lo_freq(base-diff)
#rf_module.out0_in0_lo_freq(lo_freq) """










' # module setup\nrf_module.out0_offset_path0(0)\nrf_module.out0_offset_path1(0)\nrf_module.out1_offset_path0(0)\nrf_module.out1_offset_path1(0)\n\n#rf_module.out0_lo_en(True)\n#rf_module.out1_lo_en(True)\n\nbase = 2.1e9\ndiff = 0.05e9\n\nrf_module.out0_lo_freq(base+diff)\nrf_module.out1_lo_freq(base-diff)\n#rf_module.out0_in0_lo_freq(lo_freq) '

In [88]:
# Arm and start both sequencers.
rf_module.arm_sequencer(0)
rf_module.arm_sequencer(1)
rf_module.start_sequencer()

In [84]:
# Stop both sequencers.
rf_module.stop_sequencer()

# Print status of both sequencers (should now say it is stopped).
print(rf_module.get_sequencer_status(0))
print(rf_module.get_sequencer_status(1))
print()

# Print an overview of the instrument parameters.
print("Snapshot:")
rf_module.print_readable_snapshot(update=True)

# Reset the cluster
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []

Snapshot:
cluster0_module2:
	parameter              value
--------------------------------------------------------------------------------
connected               :	True 
marker0_exp0_config     :	bypassed 
marker0_exp1_config     :	bypassed 
marker0_exp2_config     :	bypassed 
marker0_exp3_config     :	bypassed 
marker0_fir_config      :	bypassed 
marker0_inv_en          :	False 
marker1_exp0_config     :	bypassed 
marker1_exp1_config     :	bypassed 
marker1_exp2_config     :	bypassed 
marker1_exp3_config     :	bypassed 
marker1_fir_config      :	bypassed 
marker1_inv_en          :	False 
marker2_exp0_config     :	bypassed 
marker2_exp1_config     :	bypassed 
marker2_exp2_config     :	bypassed 
marker2_exp3_config     :	bypassed 
marker2_fir_config      :	bypassed 
marker2_inv_en          :	Fals